Step 1 <br>

We will be focusing on fetching youtube video's url , we have to find only Ekantik videos from Bhajan Marg youtube channel. This is done via yt-dl from terminal. read my documentation for step by step procedure or simply run 
<br>(pyTensor) tejas@Tejass-MacBook-Air Ekantik Project % yt-dlp \
  --flat-playlist \
  --dump-json \
  "https://www.youtube.com/playlist?list=PLNlLlnQWoRlndo0jrtUb6XmJJdnOaMUw5" \
  > ekantik_playlist_raw.jsonl
<br>
in terminal

Now is to filter out the fetched playlist, we have removed unnecessary data like {thumbnails,view_count,uploader / channel metadata,extractor info,epoch,availability,playlist metadata duplicates,description,live_status,version info} <br>

We want "video_id","title","url","duration_sec","declared_ekantik_number"

In [49]:
import json
import re
import os

INPUT = "ekantik_playlist_raw.jsonl"
OUTPUT = "ekantik_videos_v1.json"

if os.path.exists(OUTPUT): # to keep a check if the file already exists
    print(f"{OUTPUT} already exists.")
else:
    # Match title starting with "#<number>"
    ekantik_number_regex = re.compile(r"^\s*#\s*(\d+)")

    videos = []
    removed = 0

    with open(INPUT, "r", encoding="utf-8") as f:
        for line in f:
            raw = json.loads(line)

            title = raw.get("title")

            # Skip private / deleted
            if title in ("[Private video]", "[Deleted video]"):
                removed += 1
                continue

            # Skip inaccessible videos
            if raw.get("duration") is None:
                removed += 1
                continue

            match = ekantik_number_regex.search(title)
            declared_number = int(match.group(1)) if match else None

            videos.append({
                "video_id": raw["id"],
                "title": title,
                "url": raw.get("webpage_url"),
                "duration_sec": raw.get("duration"),
                "declared_ekantik_number": declared_number
            })

    with open(OUTPUT, "w", encoding="utf-8") as f:
        json.dump(videos, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(videos)} Ekantik videos → {OUTPUT}")
    print(f"Removed {removed} private/inaccessible videos")
    print(
        "Videos with declared Ekantik number:",
        sum(1 for v in videos if v["declared_ekantik_number"] is not None)
    )


ekantik_videos_v1.json already exists.


**Phase 2 : Fetching the Transcript**

So each of Ekantik has a  "video_id": "wGW7GKGbg9I" and "declared_ekantik_number": 21 we use this information to name each .json if we  get "declared_ekantik_number": null we will use video_id to name the file

Defining Naming convention and code to store transcript cleanly

In [50]:
def get_page_name(video_id, declared_ekantik_number):
    if declared_ekantik_number is not None:
        return f"ekantik_{declared_ekantik_number}.json"
    return f"video_{video_id}.json"


def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


To fetch transcript in hindi

In [51]:
from youtube_transcript_api import YouTubeTranscriptApi,TranscriptsDisabled,NoTranscriptFound

def fetch_hindi_transcript(video_id):
    transcript_list = YouTubeTranscriptApi().fetch(video_id, languages=["hi"])

    return transcript_list


Structure Document from fetched transcript / snippet

In [52]:
def build_page(video_id, declared_ekantik_number, snippets):
    return {
        "video_id": video_id,
        "declared_ekantik_number": declared_ekantik_number,
        "language": "hi",
        "snippets": [
            {
                "index": i + 1,
                "text": s.text,
                "start": s.start,
                "duration": s.duration,
            }
            for i, s in enumerate(snippets)
        ],
    }


Processing each page using : build_page,fetch_hindi_transcript,save_json,get_page_name

In [53]:
from pathlib import Path
import time
import random

def process_video(entry, failures):
    video_id = entry["video_id"]
    ekantik_no = entry.get("declared_ekantik_number")

    page_name = get_page_name(video_id, ekantik_no)
    output_path = Path("transcripts/hindi") / page_name


    if output_path.exists():
        print(f"already processed {page_name}")
        return  # already processed, skip

    try:
        snippets = fetch_hindi_transcript(video_id)
        page = build_page(video_id, ekantik_no, snippets)

        save_json(output_path, page)

        print(f"saved {page_name}", flush=True)

        # # ⏳ Random human-like delay: 2–5 minutes
        # sleep_seconds = random.uniform(2 * 60, 5 * 60)
        # sleep_minutes = sleep_seconds / 60
        # print(f"sleeping for {sleep_minutes:.1f} minutes...\n", flush=True)
        # time.sleep(sleep_seconds)


    except (TranscriptsDisabled, NoTranscriptFound):
        failures.append({
            "video_id": video_id,
            "declared_ekantik_number": ekantik_no,
            "reason": "Hindi transcript not available"
        })


Process all videos from our ekantik_video.json

In [54]:
def generate_transcripts(video_meta_data):
    failures = []

    for entry in video_meta_data:
        process_video(entry, failures)

    save_json(
        Path("transcripts/transcript_failures.json"),
        failures
    )


Phase 2 : Getting all the transcripts

In [55]:
with open("/Users/tejas/Documents/LangChain/ekantik_videos_v1.json", "r", encoding="utf-8") as f:
    video_meta_data = json.load(f)

generate_transcripts(video_meta_data)

already processed ekantik_456.json
already processed ekantik_477.json
already processed ekantik_478.json
already processed ekantik_479.json
already processed ekantik_614.json
already processed ekantik_641.json
already processed ekantik_999.json
already processed ekantik_1002.json
already processed ekantik_1139.json
already processed ekantik_1140.json
already processed ekantik_1141.json
already processed ekantik_1142.json
already processed ekantik_1143.json
already processed ekantik_1144.json
already processed ekantik_1145.json


Split each Ekantik 

Plan Of Action :

a. Design for singular transcript <br>
b. apply on all transcript

a.1 --> fetch a transcript from a path and return the transcript in 1 variable

In [56]:
import json
from pathlib import Path


def fetch(path):
    path = Path(path)
    with open(path, "r", encoding="utf-8") as f:
        yt_transcript_doc = json.load(f)
    
    text_transcript = "\n".join(
        snippet["text"] for snippet in yt_transcript_doc["snippets"]
    )
    meta_data = {
        "video_id": yt_transcript_doc.get("video_id"),
        "declared_ekantik_number": yt_transcript_doc.get("declared_ekantik_number"),
        "language": yt_transcript_doc.get("language"),
    }

    # text_transcript is a continious length transcript without any other data
    return text_transcript,meta_data
# text_transcript = fetch("/Users/tejas/Documents/LangChain/Ekantik Project/ekantik_1143_edited.json")
text_transcript,meta_data = fetch("/Users/tejas/Documents/LangChain/Ekantik Project/transcripts/hindi/ekantik_1107.json")


a.2 --> Creating a splitter 

In [57]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def split(text_transcript):

    split = RecursiveCharacterTextSplitter(
        chunk_size=1200,
        chunk_overlap=250,
        separators=["\n\n", "\n", "।", " ", ""] # if you see these break imideatly 
    )
    chunks= split.split_text(text_transcript)
    # These chunks or splitted trancript portions are to be converted into docs
    return chunks

In [58]:

chunks = split(text_transcript)
print(len(chunks))
print(chunks[1])

print(len(chunks[0]))
# print(len(chunks[1]))


40
लेकिन यदि किसी अन्य के द्वारा निंदा सुन
लेती हूं तो मन खिन्न हो जाता है। ऐसी
स्थिति में
दोनों का एक ही फल है। कथन और श्रवण का एक
ही फल है। यदि कोई एक आदमी किसी की निंदा
करता है तो जो उसकी दुर्गति होगी वही
सुनने वाले को ही होगी। इसलिए हमको किसी
की निंदा सुननी भी नहीं चाहिए। अगर कोई कर
रहा है तो राधा राधा राधा हमें किसी की
बात से क्या मतलब है? हमको खुद अपने आप को
सुधारना है। दूसरों के दोष सुनकर हमें
अपने हृदय को और गंदा नहीं करना है। इसलिए
पर निंदा करना और पर निंदा सुनना दोनों
पाप है। इससे बचना चाहिए।
टिकेंद्र साहू जी छत्तीसगढ़ से। श्री राधे
राधे-राधे महाराज जी महाराज जी मेरा
प्रश्न है कि श्रीमद् भगवत गीता में एक दो
बार सर्वारंभ परित्यागी शब्द कहा गया है
महाराज जी इसका अर्थ क्या है
इसका अर्थ है कि संकल्प रहित होना
जो सर्वारंभ परित्यागी संत हैं वो संकल्प
से कोई कार्य नहीं करते स्वयं हो रहा है
जैसे आदमी पहले संकल्प बनाता है फिर उस
क्रिया में उतरता है वो संत संकल्प नहीं
बनाते स्वाभाविक होता है। तत्काल होता है।
वह भगवान के प्रेमी महापुरुष होते हैं। वो
जनसाधारण नहीं होते हैं। वो सर्वारंभ
प

a.3 --> chunks converted into docs

In [59]:
from langchain_core.documents import Document

def split2docs(chunks,transcript_metadata):
    
    chunk_docs=[]
    for chunk in chunks:
        doc = Document(
            page_content=chunk,
            metadata=transcript_metadata
        )
        chunk_docs.append(doc)
    return chunk_docs

docs = split2docs(chunks,meta_data)

a.4 --> Defininf vector_store 

Defining Embedding model so later we can add our docs into it by simply doing "vector_store.add_documents(chunk_docs)"

In [1]:
from langchain_community.embeddings import HuggingFaceEmbeddings
import os
os.environ[ 'HF_HOME'] = '/Users/tejas/Documents/LangChain/Ekantik Project/embedding_model'
embedding_model = HuggingFaceEmbeddings(
   model_name="sentence-transformers/LaBSE"
)

from langchain_community.vectorstores import Chroma

vector_store = Chroma(
    embedding_function= embedding_model,
    persist_directory = "/Users/tejas/Documents/LangChain/Ekantik Project/vector_DB",# location I will store vectors
    collection_name="Ekantik_Vartalap", # name of the db / collection
)


/var/folders/q3/fl0ndk4j7v39n4nsq38qdk4w0000gn/T/ipykernel_64573/2602719828.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/opt/anaconda3/envs/pyTensor/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/q3/fl0ndk4j7v39n4nsq38qdk4w0000gn/T/ipykernel_64573/2602719828.py:10: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chr

a.5 Check if Trascripts for the video already exists 

In [2]:
def video_already_ingested(vector_store, video_id: str) -> bool:
    result = vector_store.get(
        where={"video_id": video_id},
        limit=1
    )
    return len(result["ids"]) > 0


b --> magic <br>
<br>
Time to assemble all created functions to store all the embeddings into vector db

In [ ]:
def process_transcript(path,vector_store):

    # Get video details 
    text_transcript,meta_data = fetch(path)

    #Check if the video already exists in our DB 

    vid_exist = video_already_ingested(vector_store,meta_data["video_id"])
    if(vid_exist):
        print(f"{meta_data['video_id']} already exist in our DB")
        return

    # split 
    chunks = split(text_transcript)

    # convert those chunks to docs
    docs = split2docs(chunks,meta_data)

    # save them to my vector DB
    vector_store.add_documents(docs)
    print(f"Scuessfully Stored {meta_data['declared_ekantik_number']} transcript in our db")
    

In [ ]:
path = "/Users/tejas/Documents/LangChain/Ekantik Project/transcripts/hindi/ekantik_1107.json"
process_transcript(path,vector_store)

ADDED ALL THE VIDEOS IN TRANSCRIPT 

run 

In [ ]:
from pathlib import Path

def ingest_all(transcript_dir, vector_store):
    for file in Path(transcript_dir).glob("*.json"):
        process_transcript(file, vector_store)


ingest_all("/Users/tejas/Documents/LangChain/Ekantik Project/transcripts/hindi/",vector_store)
